# Getting Started with Medical Imaging in FiftyOne

In this tutorial, we will look at how to work with Medical Imaging samples in FiftyOne — specifically DICOM files and CT scan files.

**Source:** [FiftyOne Medical Imaging Guide](https://docs.voxel51.com/getting_started/medical_imaging/01_getting_started.html)

## Installation

Install the required libraries.

In [21]:
# Dependencies are managed via pyproject.toml (uv).
# Uncomment the line below only if running outside the project environment.
# !pip install fiftyone pydicom==2.4.4 rt_utils kagglehub nibabel

## Kaggle Authentication

Set up Kaggle credentials so `kagglehub` can download datasets.

In [22]:
import os

os.environ["KAGGLE_USERNAME"] = "nnosse@wgu.edu"
os.environ["KAGGLE_KEY"] = "K4Me2Day!"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_372962a3cedc5ac0f2e8c3c0dafcf122"

print("Kaggle credentials set.")

Kaggle credentials set.


## Downloading DICOM Dataset

We'll grab an example DICOM dataset from Kaggle — the [Hippocampal Sparing Dataset](https://www.kaggle.com/datasets/aryashah2k/hippocampal-sparing-dataset).
It contains 25 brain scans with annotations of left and right hippocampus stored in `.dcm` files.

In [23]:
import kagglehub

# Download latest version
dicom_path = kagglehub.dataset_download("aryashah2k/hippocampal-sparing-dataset")

print("Path to dataset files:", dicom_path)

Path to dataset files: /Users/xnxn040/.cache/kagglehub/datasets/aryashah2k/hippocampal-sparing-dataset/versions/1


## Loading DICOM Files

FiftyOne has a built-in [DICOM dataset loader](https://docs.voxel51.com/user_guide/dataset_creation/datasets.html#dicomdataset).
This loads `.dcm` files as 2D slices that can be played back as a video once loaded.

We use `dicom_path` to select only the MR (scan) DICOMs — not the RS (annotation) DICOMs.
Let's start by loading a single patient's data.

In [24]:
import fiftyone as fo
import glob
from rt_utils import RTStructBuilder
import numpy as np
import cv2
import os


# Path to the dataset
dataset_dir = dicom_path + "/HippocampalMRISlices/01/"

# Create the dataset
dataset = fo.Dataset.from_dir(
    dataset_dir=dataset_dir,
    dataset_type=fo.types.DICOMDataset,
    dicom_path="MR*.dcm",
    name="Patient1Scans",
)

# Load RTStruct for labels
rtstruct_path = glob.glob(dataset_dir + "RS*.dcm", recursive=True)
rtstruct = RTStructBuilder.create_from(dicom_series_path=dataset_dir, rt_struct_path=rtstruct_path[0])

# Get structure names
structures = rtstruct.get_roi_names()
print("Structures in RTStruct:", structures)

# Get mask for each structure
L_mask3d = rtstruct.get_roi_mask_by_name(structures[0])
R_mask3d = rtstruct.get_roi_mask_by_name(structures[1])

# Sort dataset from bottom to top of the scan
view = dataset.sort_by("SliceLocation")

# Add masks to the dataset
i = 0
for sample in view:

    # Load left Hippocampus mask
    sample["Hippocampus_L"] = fo.Segmentation(mask=L_mask3d[:,:,i].astype(np.uint8))

    # Load right Hippocampus mask
    sample["Hippocampus_R"] = fo.Segmentation(mask=R_mask3d[:,:,i].astype(np.uint8))
    i = i + 1
    sample.save()

ValueError: Dataset name 'Patient1Scans' is not available

### Viewing the Dataset

Launch the FiftyOne App to see patient 1's scan and segmentation.

**Tip:** Use dynamic grouping in the App — group by `Patient ID` and order by `SliceLocation`,
then enable "Render frames as video" in the gear icon.

In [ ]:
session = fo.launch_app(dataset)

## Automatic Metadata Ingestion — Loading All 25 Patients

The DICOM Dataset Loader automatically extracts metadata from `.dcm` files such as
`BodyPartExamined`, `Manufacturer`, `SliceLocation`, and more.

Now let's load all 25 patients and merge them into a single dataset using `merge_samples`.

In [ ]:
# Get a list of all directories in the dataset
directories = [d for d in os.listdir(dicom_path + "/HippocampalMRISlices/") if os.path.isdir(os.path.join(dicom_path + "/HippocampalMRISlices/", d))]

# Create a new dataset to merge all the datasets
final_dataset = fo.Dataset("Hippocampal Contouring Dataset 2", overwrite=True)

# Loop through all directories
for directory in directories:
    dataset_dir = dicom_path + "/HippocampalMRISlices/" + directory + "/"

    # Create the dataset
    dataset = fo.Dataset.from_dir(
        dataset_dir=dataset_dir,
        dataset_type=fo.types.DICOMDataset,
        dicom_path="MR*.dcm"
    )

    # Load RTStruct for labels
    rtstruct_path = glob.glob(dataset_dir + "RS*.dcm", recursive=True)
    rtstruct = RTStructBuilder.create_from(dicom_series_path=dataset_dir, rt_struct_path=rtstruct_path[0])

    # Get structure names
    structures = rtstruct.get_roi_names()

    # Get mask for each structure
    L_mask3d = rtstruct.get_roi_mask_by_name(structures[0])
    R_mask3d = rtstruct.get_roi_mask_by_name(structures[1])

    # Sort dataset from bottom to top of the scan
    view = dataset.sort_by("SliceLocation")

    i = 0
    for sample in view:

        # Load left Hippocampus mask
        sample["Hippocampus_L"] = fo.Segmentation(mask=L_mask3d[:,:,i].astype(np.uint8))

        # Load right Hippocampus mask
        sample["Hippocampus_R"] = fo.Segmentation(mask=R_mask3d[:,:,i].astype(np.uint8))
        i = i + 1
        sample.save()

    # Merge the dataset with the final dataset
    final_dataset.merge_samples(dataset)

View the merged dataset with all patients:

In [ ]:
session.dataset = final_dataset

## Working with CT Scans

CT scans can come in multiple file formats: DICOM (`.dcm`), NIfTI (`.nii` or `.nii.gz`),
NRRD (`.nrrd`), or MHA/MHD (`.mha`, `.mhd` + `.raw`).

For DICOM-formatted CT scans, use the approach above. For other formats like NIfTI,
we need to slice them manually and save as images or video.

## Downloading CT Scan Data

We'll use a [COVID-19 CT Scans dataset from Kaggle](https://www.kaggle.com/datasets/andrewmvd/covid19-ct-scans).

In [ ]:
import kagglehub

# Download latest version
ct_path = kagglehub.dataset_download("andrewmvd/covid19-ct-scans")

print("Path to dataset files:", ct_path)

In [ ]:
import nibabel as nib

## Loading NIfTI Files in FiftyOne

We define helper functions to convert NIfTI files to MP4 videos (for the scan) and to
extract numpy arrays (for segmentation masks).

In [ ]:
def nii_to_mp4(filepath, fps=30):
    '''
    Reads .nii file and returns pixel array
    '''
    # Load .nii file
    ct_scan = nib.load(filepath)
    array   = ct_scan.get_fdata()

    # Rotate array
    array   = np.rot90(np.array(array))

    # Normalize pixel values
    array = (array - np.min(array)) / (np.max(array) - np.min(array)) * 255
    array = array.astype(np.uint8)

    # Define video writer
    height, width, depth = array.shape
    output_path = filepath.replace(".nii", ".mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

    # Write each axial slice to the video
    for i in range(depth):
        frame = array[:, :, i]
        # Convert grayscale to BGR for consistent writing
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        video_writer.write(frame_bgr)

    video_writer.release()
    return(output_path)

In [ ]:
def read_nii(filepath):
    '''
    Reads .nii file and returns pixel array
    '''
    ct_scan = nib.load(filepath)
    array   = ct_scan.get_fdata()
    array   = np.rot90(np.array(array))
    array = array.astype(np.uint8)
    return(array)

### Load a Single CT Scan Patient

In [ ]:
# Convert Scan to mp4
sample_ct   = nii_to_mp4(ct_path + "/ct_scans/coronacases_org_001.nii")

# Read the masks
sample_lung = read_nii(ct_path + "/lung_mask/coronacases_001.nii")
sample_infe = read_nii(ct_path + "/infection_mask/coronacases_001.nii")
sample_all  = read_nii(ct_path + "/lung_and_infection_mask/coronacases_001.nii")

Create a FiftyOne video sample and attach the segmentation masks to each frame.
Note that video frames in FiftyOne are 1-indexed (there is no frame 0).

In [ ]:
# Create video sample
sample = fo.Sample(filepath=sample_ct)


for i in range(sample_lung.shape[2]):

    # Add masks to each frame. Note that video frames start from 1, there is no frame 0!
    sample.frames[i+1]["Infection Mask"] = fo.Segmentation(mask=sample_infe[:, :, i].astype(np.uint8))
    sample.frames[i+1]["Lung Mask"] = fo.Segmentation(mask=sample_lung[:, :, i].astype(np.uint8))
    sample.frames[i+1]["All Masks"] = fo.Segmentation(mask=sample_all[:, :, i].astype(np.uint8))

### Create the Dataset and Re-encode Video

We use `fo.utils.video.reencode_videos()` to ensure the MP4 is browser-compatible.

In [ ]:
# Create a dataset of Patient 1
dataset = fo.Dataset(name="COVID-19 CT Scans Patient 1", overwrite=True, media_type="video")
dataset.add_samples([sample])

# Reencode videos so it will play in any browser
fo.utils.video.reencode_videos(dataset)

# Launch the App
session = fo.launch_app(dataset)

## Summary

In this notebook we covered:

1. **Loading DICOM files** into FiftyOne with automatic metadata extraction
2. **Adding segmentation masks** from RTStruct annotations
3. **Merging multiple patients** into a single dataset
4. **Working with CT scans** in NIfTI format by converting to MP4 video
5. **Attaching frame-level segmentation masks** to video samples

For more details, visit the [FiftyOne Medical Imaging Guide](https://docs.voxel51.com/getting_started/medical_imaging/01_getting_started.html).